# Задание 2: мини-autograd (класс `Node`)

Реализован мини-фреймворк автоматического дифференцирования: маленькая библиотека на скалярных значениях, которая сама умеет считать градиенты для сложения, умножения и ReLU.

**Как работает:** Каждое значение оборачивается в объект `Node`.
Когда мы выполняем операцию (`+`, `*`, `relu`), создаётся новый узел, который
запоминает, из каких узлов он получился, и как пересчитать градиент для своих
«родителей». Вызов `backward()` на итоговом узле:
1. Строит топологический порядок всего графа
2. Запускает локальные «обратные» функции в обратном порядке
3. Суммирует градиенты в каждом узле

In [1]:
import unittest

## Класс `Node`

Поля:
* `data` — значение узла
* `grad` — накопленный градиент (по умолчанию 0)
* `_backward` — функция, которая умеет «протолкнуть» градиент от этого узла к узлам, из которых он построен
* `_prev` — множество узлов-родителей
* `_op` — название операции (для отладки)

In [ ]:
class Node:
    """Узел вычислительного графа: хранит значение и градиент"""

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Node(data={self.data}, grad={self.grad})"

    # Сложение
    def __add__(self, other):
        # Поддерживаем "узел + число": число оборачиваем в Node
        if not isinstance(other, Node):
            other = Node(other)
        out = Node(self.data + other.data, (self, other), '+')

        def _backward():
            # d(out)/d(self) = 1 и d(out)/d(other) = 1: просто пропускаем градиент дальше
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __radd__(self, other):
        # Чтобы работало "число + узел"
        return self + other

    # Умножение
    def __mul__(self, other):
        if not isinstance(other, Node):
            other = Node(other)
        out = Node(self.data * other.data, (self, other), '*')

        def _backward():
            # d(out)/d(self) = other.data, d(out)/d(other) = self.data
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __rmul__(self, other):
        # Чтобы работало "число * узел"
        return self * other

    # Активация ReLU
    def relu(self):
        out = Node(self.data if self.data > 0 else 0, (self,), 'relu')

        def _backward():
            # Производная ReLU: 1 при data > 0, иначе 0
            if self.data > 0:
                self.grad += out.grad

        out._backward = _backward
        return out

    # Обратный проход
    def backward(self):
        # 1. Топологическая сортировка: узлы идут так, что родители идут после детей
        topo = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for child in node._prev:
                    build_topo(child)
                topo.append(node)

        build_topo(self)

        # 2. Начинаем с градиента 1 для самого узла и идём в обратном порядке
        self.grad = 1
        for node in reversed(topo):
            node._backward()

    # Небольшой помощник для тестов
    def zero_grad(self):
        """Обнуляет градиенты этого узла и всех узлов, из которых он построен"""
        visited = set()
        stack = [self]
        while stack:
            node = stack.pop()
            if node not in visited:
                visited.add(node)
                node.grad = 0
                for child in node._prev:
                    stack.append(child)

## Демонстрация: пример из задания

Обращаем внимание на аккуратный разбор этого примера:
`d = a + b * c = 2 + (-3)*10 = -28`. Поскольку `-28 < 0`, ReLU переводит значение в 0,
и его производная в этой точке равна нулю - значит, градиент «закрывается» и до узлов
`a`, `b`, `c` не доходит. Единичный градиент остаётся только у `e`.

In [3]:
a = Node(2)
b = Node(-3)
c = Node(10)
d = a + b * c
e = d.relu()
e.backward()
print(a, b, c, d, e)

Node(data=2, grad=0) Node(data=-3, grad=0) Node(data=10, grad=0) Node(data=-28, grad=0) Node(data=0, grad=1)


### Пример, где ReLU активен: градиенты проходят через все узлы

In [ ]:
a = Node(2)
b = Node(-3)
c = Node(-10)
d = a + b * c      # 2 + 30 = 32 (положительное число)
e = d.relu()       # ReLU(32) = 32
e.backward()
print(a, b, c, d, e)

# Проверка вручную: d = a + b*c
#   dd/da = 1          -> a.grad = 1
#   dd/db = c = -10    -> b.grad = -10
#   dd/dc = b = -3     -> c.grad = -3

Node(data=2, grad=1) Node(data=-3, grad=-10) Node(data=-10, grad=-3) Node(data=32, grad=1) Node(data=32, grad=1)


### Пример подлиннее: несколько операций в одной цепочке

In [ ]:
x = Node(3)
y = Node(4)
z = x * y          # 12
z = z.relu()       # 12
z = z + x          # 15
z.backward()
print(x, y, z)

# z = ReLU(x*y) + x, поэтому:
#   dz/dx = y + 1 = 5
#   dz/dy = x = 3

Node(data=3, grad=5) Node(data=4, grad=3) Node(data=15, grad=1)


## Тесты (unittest)

Тесты покрывают: сложение и умножение (значения и градиенты), ReLU во всех режимах,
цепочки операций, повторное использование одного узла (накопление градиентов),
смешанные типы int/float, операторы с числом слева, а также численную проверку
градиентов методом конечных разностей.

In [ ]:
class TestAddition(unittest.TestCase):
    def test_forward_value(self):
        x = Node(3)
        y = Node(4)
        z = x + y
        self.assertEqual(z.data, 7)

    def test_gradients(self):
        x = Node(3)
        y = Node(4)
        z = x + y
        z.backward()
        self.assertEqual(x.grad, 1)
        self.assertEqual(y.grad, 1)

    def test_add_number(self):
        x = Node(5)
        z = x + 2
        z.backward()
        self.assertEqual(z.data, 7)
        self.assertEqual(x.grad, 1)

    def test_radd_number(self):
        x = Node(5)
        z = 2 + x
        z.backward()
        self.assertEqual(z.data, 7)
        self.assertEqual(x.grad, 1)


class TestMultiplication(unittest.TestCase):
    def test_forward_value(self):
        x = Node(3)
        y = Node(-2)
        z = x * y
        self.assertEqual(z.data, -6)

    def test_gradients(self):
        x = Node(3)
        y = Node(-2)
        z = x * y
        z.backward()
        self.assertEqual(x.grad, -2)
        self.assertEqual(y.grad, 3)

    def test_mul_number(self):
        x = Node(4)
        z = x * 3
        z.backward()
        self.assertEqual(z.data, 12)
        self.assertEqual(x.grad, 3)

    def test_rmul_number(self):
        x = Node(4)
        z = 3 * x
        z.backward()
        self.assertEqual(z.data, 12)
        self.assertEqual(x.grad, 3)


class TestReLU(unittest.TestCase):
    def test_positive(self):
        x = Node(5)
        y = x.relu()
        y.backward()
        self.assertEqual(y.data, 5)
        self.assertEqual(x.grad, 1)

    def test_negative(self):
        x = Node(-5)
        y = x.relu()
        y.backward()
        self.assertEqual(y.data, 0)
        self.assertEqual(x.grad, 0)

    def test_zero(self):
        x = Node(0)
        y = x.relu()
        y.backward()
        self.assertEqual(y.data, 0)
        self.assertEqual(x.grad, 0)


class TestChains(unittest.TestCase):
    def test_example_from_task(self):
        # Пример из задания; корректные значения градиентов
        a = Node(2)
        b = Node(-3)
        c = Node(10)
        d = a + b * c
        e = d.relu()          # d = -28 < 0 -> ReLU закрывает градиент
        e.backward()
        self.assertEqual(e.data, 0)
        self.assertEqual(e.grad, 1)
        self.assertEqual(d.grad, 0)
        self.assertEqual(a.grad, 0)
        self.assertEqual(b.grad, 0)
        self.assertEqual(c.grad, 0)

    def test_chain_active_relu(self):
        a = Node(2)
        b = Node(-3)
        c = Node(-10)
        d = a + b * c         # 32 > 0 -> градиент проходит
        e = d.relu()
        e.backward()
        self.assertEqual(e.data, 32)
        self.assertEqual(a.grad, 1)
        self.assertEqual(b.grad, -10)
        self.assertEqual(c.grad, -3)

    def test_shared_node_accumulates_gradients(self):
        # y = x*x + x -> dy/dx = 2x + 1 = 7
        x = Node(3)
        y = x * x + x
        y.backward()
        self.assertEqual(y.data, 12)
        self.assertEqual(x.grad, 7)

    def test_deep_chain(self):
        # z = ReLU(x*x) + 2*x -> dz/dx = 2x + 2 = 5 при x = 1.5
        x = Node(1.5)
        z = (x * x).relu() + x * 2
        z.backward()
        self.assertAlmostEqual(z.data, 2.25 + 3.0)
        self.assertAlmostEqual(x.grad, 5.0)

    def test_mixed_int_float(self):
        # z = (x + y) * x -> dz/dx = 2x + y, dz/dy = x
        x = Node(2)
        y = Node(0.5)
        z = (x + y) * x
        z.backward()
        self.assertAlmostEqual(z.data, 5.0)
        self.assertAlmostEqual(x.grad, 4.5)
        self.assertAlmostEqual(y.grad, 2.0)


def relu_expr_grads(x, y, z):
    """f(x, y, z) = relu(x*y + z). Возвращает (значение, df/dx, df/dy, df/dz)"""
    nx = Node(x)
    ny = Node(y)
    nz = Node(z)
    out = (nx * ny + nz).relu()
    out.backward()
    return out.data, nx.grad, ny.grad, nz.grad


def relu_expr_value(x, y, z):
    """То же выражение, но без autograd - для численной проверки"""
    s = x * y + z
    return s if s > 0 else 0


class TestNumericalGradients(unittest.TestCase):
    """Сравниваем аналитические градиенты с конечными разностями (f(x+h)-f(x-h))/(2h)"""

    def check_point(self, x, y, z):
        eps = 1e-5
        _, gx, gy, gz = relu_expr_grads(x, y, z)

        num_gx = (relu_expr_value(x + eps, y, z) - relu_expr_value(x - eps, y, z)) / (2 * eps)
        num_gy = (relu_expr_value(x, y + eps, z) - relu_expr_value(x, y - eps, z)) / (2 * eps)
        num_gz = (relu_expr_value(x, y, z + eps) - relu_expr_value(x, y, z - eps)) / (2 * eps)

        self.assertAlmostEqual(gx, num_gx, places=5)
        self.assertAlmostEqual(gy, num_gy, places=5)
        self.assertAlmostEqual(gz, num_gz, places=5)

    def test_active_zone(self):
        # x*y + z = 10 > 0: ReLU активен
        self.check_point(2.0, 3.0, 4.0)

    def test_inactive_zone(self):
        # x*y + z = -4 < 0: ReLU закрыт, все градиенты нулевые
        self.check_point(1.0, 1.0, -5.0)

    def test_two_points(self):
        # ещё одна отрицательная точка и положительная
        self.check_point(-2.0, 3.0, -7.0)
        self.check_point(-2.0, 3.0, 10.0)

### Запуск всех тестов

In [7]:
loader = unittest.TestLoader()
suite = unittest.TestSuite()
suite.addTests(loader.loadTestsFromTestCase(TestAddition))
suite.addTests(loader.loadTestsFromTestCase(TestMultiplication))
suite.addTests(loader.loadTestsFromTestCase(TestReLU))
suite.addTests(loader.loadTestsFromTestCase(TestChains))
suite.addTests(loader.loadTestsFromTestCase(TestNumericalGradients))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print()
print("Все тесты прошли успешно:", result.wasSuccessful())
print("Количество тестов:", result.testsRun)

test_add_number (__main__.TestAddition.test_add_number) ... 

ok


test_forward_value (__main__.TestAddition.test_forward_value) ... 

ok


test_gradients (__main__.TestAddition.test_gradients) ... 

...

ok


test_two_points (__main__.TestNumericalGradients.test_two_points) ... 

ok


----------------------------------------------------------------------
Ran 19 tests in 0.028s

OK



Все тесты прошли успешно: True
Количество тестов: 19
